# Module 4 — Mini Project & Mini Assignment

## Mini Project — Enterprise Knowledge Assistant

Installs pin to specific `numpy`/`scipy`/`scikit-learn`/`sentence-transformers` versions so `ConversationalRetrievalChain` and `EnsembleRetriever` behave predictably.

In [ ]:
!pip uninstall -y numpy scipy scikit-learn sentence-transformers
!pip install -q \
numpy==1.26.4 \
scipy==1.13.1 \
scikit-learn==1.5.1 \
sentence-transformers==3.0.1

### Upload one or more source PDFs

In [1]:
from google.colab import files

uploaded_files = files.upload()
pdf_filenames = list(uploaded_files.keys())
pdf_filenames

['The Ultimate Python Handbook (1).pdf']


### Load every uploaded PDF, keeping source + page number in metadata

In [1]:
from langchain_community.document_loaders import PyPDFLoader

knowledge_docs = []
for fname in pdf_filenames:
    knowledge_docs.extend(PyPDFLoader(fname).load())

knowledge_docs

[Document(metadata={'source': 'The Ultimate Python Handbook (1).pdf', 'page': 0}, page_content=''),
 Document(metadata={'source': 'The Ultimate Python Handbook (1).pdf', 'page': 1}, page_content='PREFACE \nWelcome to the "Ultimate Python Programming Handbook," your comprehensive guide to mastering Python programming...'),
 Document(metadata={'source': 'The Ultimate Python Handbook (1).pdf', 'page': 2}, page_content='CONTENTS \nPREFACE ... Chapter 1 – Modules, Comments & pip ...'),
 ... 57 more Document objects, one per remaining page of the handbook ...]


### Chunk every page into smaller, overlapping pieces

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
knowledge_chunks = splitter.split_documents(knowledge_docs)
knowledge_chunks

[Document(metadata={'source': 'The Ultimate Python Handbook (1).pdf', 'page': 1}, page_content='PREFACE \nWelcome to the "Ultimate Python Programming Handbook," your comprehensive guide to mastering Python programming. This handbook is designed for beginners...'),
 Document(metadata={'source': 'The Ultimate Python Handbook (1).pdf', 'page': 1}, page_content='PURPOSE AND AUDIENCE ...'),
 ... several hundred more 300-character chunks, each tagged with its source page ...]


### Build a sparse (BM25) retriever, a dense (FAISS) retriever, and combine them

In [1]:
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.retrievers import EnsembleRetriever

sparse_retriever = BM25Retriever.from_documents(knowledge_chunks)
sparse_retriever.k = 3

embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_store = FAISS.from_documents(knowledge_chunks, embedder)
dense_retriever = dense_store.as_retriever(search_kwargs={"k": 3})

hybrid_retriever = EnsembleRetriever(
    retrievers=[sparse_retriever, dense_retriever],
    weights=[0.5, 0.5],
)

### Load a Hugging Face model and set up conversation memory

In [1]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.memory import ConversationBufferMemory

gen_pipeline = pipeline("text2text-generation", model="google/flan-t5-base", max_new_tokens=100)
llm = HuggingFacePipeline(pipeline=gen_pipeline)

chat_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

Device set to use cpu


### Wire the hybrid retriever + LLM + memory into a conversational chain

In [ ]:
from langchain.chains import ConversationalRetrievalChain

knowledge_assistant = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=hybrid_retriever,
    memory=chat_memory,
    return_source_documents=True,
)

### Ask an initial question, then a follow-up, and inspect the citations for each

In [1]:
q1 = "What is this document about?"
r1 = knowledge_assistant.invoke({"question": q1})

print("Question:", q1)
print("Answer:", r1["answer"])
print("Citations:")
for doc in r1["source_documents"]:
    print(f"- {doc.metadata.get('source')} (page {doc.metadata.get('page')})")
print("-" * 80)

q2 = "Can you explain that in more detail?"
r2 = knowledge_assistant.invoke({"question": q2})

print("Question:", q2)
print("Answer:", r2["answer"])
print("Citations:")
for doc in r2["source_documents"]:
    print(f"- {doc.metadata.get('source')} (page {doc.metadata.get('page')})")

Question: What is this document about?
Answer: PURPOSE AND AUDIENCE This handbook aims to make programming accessible and enjoyable for everyone. Whether you're a student new to coding, a professional seeking to enhance your skills, or an enthusiast...
Citations:
- The Ultimate Python Handbook (1).pdf (page 2)
- The Ultimate Python Handbook (1).pdf (page 1)
- The Ultimate Python Handbook (1).pdf (page 22)
- The Ultimate Python Handbook (1).pdf (page 1)
- The Ultimate Python Handbook (1).pdf (page 16)
- The Ultimate Python Handbook (1).pdf (page 4)
--------------------------------------------------------------------------------
Question: Can you explain that in more detail?
Answer: (follow-up answer grounded in the same retrieved pages)
Citations:
- The Ultimate Python Handbook (1).pdf (page 2)
- The Ultimate Python Handbook (1).pdf (page 14)
- The Ultimate Python Handbook (1).pdf (page 22)
- The Ultimate Python Handbook (1).pdf (page 11)
- The Ultimate Python Handbook (1).pdf (page 22)

### Inspect the conversation history the assistant retained

In [1]:
chat_memory.chat_memory.messages

[HumanMessage(content='What is this document about?', ...),
 AIMessage(content="PURPOSE AND AUDIENCE This handbook aims to make programming accessible and enjoyable for everyone...", ...),
 HumanMessage(content='Can you explain that in more detail?', ...),
 AIMessage(content='(follow-up answer)', ...)]


## Mini Assignment — Comparing RAG Architectures

### Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers pandas

### Build a tiny Standard RAG pipeline: write, chunk, and embed a note

In [1]:
rag_note = (
    "Standard RAG embeds a user question, retrieves the most relevant chunks from a vector "
    "database, and passes them to a language model to generate a grounded answer. It is "
    "simple and fast but only performs a single retrieval step, so it struggles with "
    "complex, multi-step, or highly personalized questions."
)

with open("standard_rag_notes.txt", "w") as f:
    f.write(rag_note)

with open("standard_rag_notes.txt", "r") as f:
    loaded_note = f.read()

note_chunks = [piece.strip() for piece in loaded_note.split(". ") if piece.strip()]

from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("all-MiniLM-L6-v2")
note_embeddings = st_model.encode(note_chunks)
note_embeddings.shape

(2, 384)


### Index the note, load a generator, and answer a sample question

In [1]:
import faiss
import numpy as np
from transformers import pipeline

vec_dim = note_embeddings.shape[1]
note_index = faiss.IndexFlatL2(vec_dim)
note_index.add(np.array(note_embeddings))

answer_generator = pipeline("text2text-generation", model="google/flan-t5-base")

sample_q = "What does Standard RAG struggle with?"
q_vec = st_model.encode([sample_q])
top_dists, top_idx = note_index.search(np.array(q_vec), 2)

retrieved = " ".join(note_chunks[idx] for idx in top_idx[0])
rag_prompt = f"Answer the question using the context.\n\nContext: {retrieved}\n\nQuestion: {sample_q}"
generated_answer = answer_generator(rag_prompt, max_new_tokens=100)[0]["generated_text"]

print("Question:", sample_q)
print("Retrieved Context:", retrieved)
print("Generated Answer:", generated_answer)

Device set to use cpu
Question: What does Standard RAG struggle with?
Retrieved Context: Standard RAG embeds a user question, retrieves the most relevant chunks from a vector database, and passes them to a language model to generate a grounded answer It is simple and fast but only performs a single retrieval step, so it struggles with complex, multi-step, or highly personalized questions.
Generated Answer: complex, multi-step, or highly personalized questions


### Compare Standard RAG against Agentic, Memory, and Self RAG

In [ ]:
import pandas as pd

architecture_comparison = pd.DataFrame({
    "Architecture": ["Standard RAG", "Agentic RAG", "Memory RAG", "Self RAG"],
    "Accuracy": [
        "Moderate - bounded by a single retrieval pass",
        "Higher on complex, multi-step questions since the agent can plan and use tools",
        "Higher for personalized or ongoing conversations; similar to Standard on a first question",
        "Highest - the answer is checked and regenerated if self-evaluation flags it as weak",
    ],
    "Retrieval Quality": [
        "Depends entirely on a single retrieval step",
        "Improved via multiple planned retrieval steps and tool calls",
        "Same underlying retrieval as Standard, enriched with conversation context",
        "Improved iteratively - retrieval repeats if the first pass is judged poor",
    ],
    "Response Time": [
        "Fastest - one retrieval call plus one generation call",
        "Slower - multiple agent reasoning and tool-call steps add latency",
        "Close to Standard, slightly higher due to memory lookup",
        "Slowest - may retrieve and generate more than once before finalizing",
    ],
    "Hallucination Rate": [
        "Higher on complex or ambiguous questions",
        "Lower - multi-step reasoning and tool verification catch more errors",
        "Lower on follow-ups since prior context carries over; similar to Standard on turn one",
        "Lowest - the self-evaluation step catches ungrounded answers before returning them",
    ],
})

architecture_comparison